# Security and compliance

Screen outbound text, access reasons, policy clauses, odd audit events, and a pair of emails. The fake key in the fixtures is not a real secret.


In [ ]:
import sys
from datetime import date
from pathlib import Path
import json
import re
import statistics

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from langchain_typesafe import Choice, Noul, NoulCriteria, Score
from jev_examples.settings import ask, ask_many, draft, jev_model, openai_ready, show, typesafe_ready
from jev_examples.sample_data import (
    corpus_docs,
    customers,
    emails,
    load_json,
    lookup_order,
    open_incidents,
    order,
    products,
    read_text,
    ticket,
    tickets,
)

print("Jev model:", jev_model())
print("Jev key set:", typesafe_ready())
print("OpenAI key set:", openai_ready())


## 49. Outbound data check

Block a send that carries a shopper record or a key. Let a plain shipping update through.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "personal_data": Noul(instructions="Does `message` contain contact details of an identifiable shopper?"),
        "credential": Noul(instructions="Does `message` contain a secret, token, or password?"),
        "severity": Score(
            instructions="If `message` were sent outside the company, how bad would it be?",
            criteria=["Fine", "Embarrassing", "Policy breach"],
        ),
    }
    for text in load_json("messages.json")["outbound"]:
        response = ask({"message": text}, questions)
        show(response)
        if any(answer.noul > 0.55 for answer in response.nouls.values()) or response.scores["severity"].score > 1.4:
            route = "block"
        else:
            route = "send"
        print("route:", route)


**What you should see.** The tracking sentence should send. The message with Maya's email and the test key should block.


## 50. Does the reason match the access being asked for?

A one-week read of orders is a different request from curiosity about the admin page.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "fit": Score(
            instructions="How well does `justification` explain the need for `requested_scope`?",
            criteria=["Unrelated", "Partial", "Clear"],
        ),
        "narrower_would_do": Noul(instructions="Could the need be met by a narrower scope than `requested_scope`?"),
        "time_bound": Noul(instructions="Does `justification` describe a temporary need?"),
    }
    for item in load_json("messages.json")["access_requests"]:
        response = ask(item, questions)
        show(response)
        fit = response.scores["fit"]
        if fit.score > 1.4 and fit.confidence > 0.55 and response.nouls["narrower_would_do"].noul < 0.45:
            route = "approve"
            if response.nouls["time_bound"].noul > 0.55:
                route = "approve_temporary"
        else:
            route = "reviewer_queue"
        print(item["requested_scope"], "->", route)


**What you should see.** The camp refund read should approve, likely as temporary. Admin-out-of-curiosity should go to a reviewer.


## 51. Which policy clause governs this action?

Map the action to a clause, then ask if it is allowed without extra approval.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    clauses = dict(load_json("messages.json")["clauses"])
    clauses["none"] = "No clause applies"
    response = ask(
        {"action": "Refund a wholesale tent order placed 10 days ago."},
        {
            "clause": Choice(instructions="Which policy clause governs `action`?", criteria=clauses),
            "permitted": Noul(instructions="Under the applicable clause, is `action` allowed without extra approval?"),
        },
    )
    show(response)
    allowed = response.nouls["permitted"].noul > 0.65 and response.choices["clause"].confidence > 0.5
    print(response.choices["clause"].choice, "allowed without approval:", allowed)


**What you should see.** The returns clause should be selected. Wholesale returns in the contract need written approval, so 'allowed without approval' should be false or low.


## 52. Flag an audit event that does not match the role

Support reads orders. Support does not export the customer table from payroll.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    profile = load_json("messages.json")["role_profile"]
    questions = {
        "consistent_with_role": Noul(instructions="Is `event` the kind of action described as normal in `role_profile`?"),
        "unusual_target": Noul(instructions="Does the event target a system outside the ones a support agent uses?"),
    }
    events = load_json("messages.json")["audit_events"]
    requests = [{"state": {"event": event, "role_profile": profile}, "questions": questions} for event in events]
    for event, response in zip(events, ask_many(requests)):
        show(response)
        flag = response.nouls["consistent_with_role"].noul < 0.4 or response.nouls["unusual_target"].noul > 0.6
        print(event["action"], "->", "flag" if flag else "ok")


**What you should see.** Reading A-118 should be ok. Exporting the customer table toward payroll should be flagged.


## 53. Phishing as several small questions

A prize email that asks for a password is not one vague 'is this phish?' score. Each signal is its own Noul. Python adds them up.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "requests_credentials": Noul(instructions="Does `body` ask the recipient to provide a password or other login secret?"),
        "unexpected_reward": Noul(instructions="Does `body` claim an unexpected prize or payment?"),
        "time_pressure": Noul(instructions="Does `subject` or `body` pressure the recipient to act quickly?"),
        "sender_mismatch": Noul(instructions="Does the organization in `sender.display_name` conflict with the domain in `sender.email`?"),
        "link_mismatch": Noul(instructions="Does the domain in the first link conflict with `sender.display_name`?"),
    }
    for message in load_json("messages.json")["phishing"]:
        response = ask({"message": message}, questions)
        show(response)
        score = (
            0.35 * response.nouls["requests_credentials"].noul
            + 0.2 * response.nouls["sender_mismatch"].noul
            + 0.2 * response.nouls["link_mismatch"].noul
            + 0.15 * response.nouls["unexpected_reward"].noul
            + 0.1 * response.nouls["time_pressure"].noul
        )
        print(round(score, 2), "->", "review" if score > 0.45 else "inbox")


**What you should see.** The prize email from claim-bonus.example should review. The real reset note from northwind.example should stay in the inbox.
